# 02 — Sanity Check Model

Notebook ini memverifikasi:
- Model MAE-ViT backbone bisa di-load
- Forward pass dummy berjalan tanpa error
- Output shape sesuai harapan
- LoRA bisa di-attach dengan benar
- Jumlah parameter trainable sesuai

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ahmdzaidan/cassava-lora.git
%cd cassava-lora

Mounted at /content/drive
Cloning into 'cassava-lora'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 135 (delta 36), reused 123 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 3.93 MiB | 7.55 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/cassava-lora


In [2]:
# Install Requirements
!pip install -q -r /content/cassava-lora/requirements.txt
!python -c "import torch, transformers, peft; print('OK')"
!python -c "import torch; print('GPU:', torch.cuda.is_available())"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.3.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client

In [3]:
import sys
sys.path.insert(0, '..')

import torch
from src.utils.config import load_config
from src.utils.seed import set_seed

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

[seed.py] Global seed set to 42
Device: cuda


## 1. Load Backbone

In [4]:
from src.models.mae_vit import load_mae_vit_backbone, freeze_backbone, get_backbone_info

model_config = load_config('./configs/model_mae_vit.yaml')

backbone = load_mae_vit_backbone(
    model_name=model_config['backbone']['model_name'],
    pretrained=True,
    device=device
)

info = get_backbone_info(backbone)
print(f'\nBackbone Info:')
for k, v in info.items():
    print(f'  {k}: {v}')

[mae_vit.py] Loading backbone: facebook/vit-mae-base
[mae_vit.py] Pretrained: True
[mae_vit.py] Device: cuda


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  448MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTMAEModel LOAD REPORT from: facebook/vit-mae-base
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_before.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.o_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc2.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.v_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_after.bias    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_after.weight  | UNEXPECTED |  | 
decoder.decoder_layers.

[mae_vit.py] Weights transferred: 198/200
[mae_vit.py] Missing keys (randomly initialized): 2
[mae_vit.py] Total parameters: 86,389,248
[mae_vit.py] Hidden size: 768
[mae_vit.py] Num layers: 12
[mae_vit.py] Num attention heads: 12

Backbone Info:
  total_params: 86389248
  trainable_params: 86389248
  frozen_params: 0
  trainable_pct: 100.0
  hidden_size: 768
  num_layers: 12


## 2. Forward Pass Dummy

In [5]:
# Dummy input: batch of 4 images, 3 channels, 224x224
dummy_input = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    output = backbone(pixel_values=dummy_input, output_hidden_states=True)

print(f'Last hidden state shape: {output.last_hidden_state.shape}')
print(f'CLS token shape: {output.last_hidden_state[:, 0, :].shape}')
print(f'Num hidden states: {len(output.hidden_states)}')

# Expected: (4, 197, 768) — 196 patches + 1 CLS token
assert output.last_hidden_state.shape == (4, 197, 768), 'Shape mismatch!'
print('\n✓ Forward pass successful!')

Last hidden state shape: torch.Size([4, 197, 768])
CLS token shape: torch.Size([4, 768])
Num hidden states: 13

✓ Forward pass successful!


## 3. Classifier Head

In [6]:
from src.models.classifier_head import CassavaClassifier, build_classifier_head

head_config = model_config['classifier'].copy()
head_config['hidden_size'] = 768
head = build_classifier_head(head_config).to(device)

model = CassavaClassifier(backbone, head).to(device)

with torch.no_grad():
    result = model(pixel_values=dummy_input)

print(f'Logits shape: {result["logits"].shape}')
print(f'CLS embedding shape: {result["cls_embedding"].shape}')

assert result['logits'].shape == (4, 5), 'Logits shape mismatch!'
print('\n✓ Classifier head works!')

Logits shape: torch.Size([4, 5])
CLS embedding shape: torch.Size([4, 768])

✓ Classifier head works!


## 4. LoRA Attachment

In [10]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [11]:
from src.models.lora_layers import attach_lora_peft, extract_lora_weights_peft
from src.models.mae_vit import load_mae_vit_backbone, freeze_backbone

# Fresh backbone for LoRA test
backbone_lora = load_mae_vit_backbone(device=device)
freeze_backbone(backbone_lora)

# Attach LoRA
backbone_lora = attach_lora_peft(
    backbone_lora, rank=8, alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05
)

# Forward pass
with torch.no_grad():
    output = backbone_lora(pixel_values=dummy_input)

print(f'\nOutput shape: {output.last_hidden_state.shape}')
print('\n✓ LoRA attachment successful!')

[mae_vit.py] Loading backbone: facebook/vit-mae-base
[mae_vit.py] Pretrained: True
[mae_vit.py] Device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTMAEModel LOAD REPORT from: facebook/vit-mae-base
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_before.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.o_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc2.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.v_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_after.bias    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_after.weight  | UNEXPECTED |  | 
decoder.decoder_layers.

[mae_vit.py] Weights transferred: 198/200
[mae_vit.py] Missing keys (randomly initialized): 2
[mae_vit.py] Total parameters: 86,389,248
[mae_vit.py] Hidden size: 768
[mae_vit.py] Num layers: 12
[mae_vit.py] Num attention heads: 12
[mae_vit.py] Frozen: 200/200 parameter tensors
trainable params: 294,912 || all params: 86,684,160 || trainable%: 0.3402

Output shape: torch.Size([4, 197, 768])

✓ LoRA attachment successful!


In [12]:
# Extract LoRA weights
lora_weights = extract_lora_weights_peft(backbone_lora)

print(f'Number of LoRA layers: {len(lora_weights)}')
for name, weights in list(lora_weights.items())[:3]:
    print(f'  {name}:')
    print(f'    A shape: {weights["A"].shape}')
    print(f'    B shape: {weights["B"].shape}')
    print(f'    ΔW shape: {weights["delta_W"].shape}')

[lora_layers.py] Extracted LoRA weights from 24 layers
Number of LoRA layers: 24
  base_model.model.layers.0.attention.q_proj:
    A shape: torch.Size([8, 768])
    B shape: torch.Size([768, 8])
    ΔW shape: torch.Size([768, 768])
  base_model.model.layers.0.attention.v_proj:
    A shape: torch.Size([8, 768])
    B shape: torch.Size([768, 8])
    ΔW shape: torch.Size([768, 768])
  base_model.model.layers.1.attention.q_proj:
    A shape: torch.Size([8, 768])
    B shape: torch.Size([768, 8])
    ΔW shape: torch.Size([768, 768])
